# Product Analytics: E-commerce Funnel, Retention and A/B Test

Pet project for Product Analyst / Data Analyst Intern applications.

In [ ]:
from pathlib import Path
import math
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(Path('../data/events_sample.csv'), parse_dates=['event_timestamp', 'event_date'])
df.head()

## Product funnel

In [ ]:
steps = ['session_start', 'view_item', 'add_to_cart', 'begin_checkout', 'purchase']
funnel = df[df.event_name.isin(steps)].groupby('event_name')['user_id'].nunique().reindex(steps).reset_index()
funnel.columns = ['step', 'users']
funnel['step_conversion'] = funnel.users / funnel.users.shift(1)
funnel.loc[0, 'step_conversion'] = 1.0
funnel['overall_conversion'] = funnel.users / funnel.loc[0, 'users']
funnel

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(funnel['step'], funnel['users'])
plt.title('Product funnel: users by step')
plt.xlabel('Funnel step')
plt.ylabel('Unique users')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## Metrics

In [ ]:
purchases = df[df.event_name == 'purchase']
metrics = {
    'users': df.user_id.nunique(),
    'sessions': df.session_id.nunique(),
    'orders': purchases.order_id.nunique(),
    'revenue': purchases.revenue.sum(),
    'AOV': purchases.revenue.sum() / purchases.order_id.nunique(),
    'ARPU': purchases.revenue.sum() / df.user_id.nunique(),
    'ARPPU': purchases.revenue.sum() / purchases.user_id.nunique(),
}
pd.Series(metrics)

## Retention

In [ ]:
first_seen = df.groupby('user_id')['event_date'].min().rename('first_date')
activity = df[['user_id', 'event_date']].drop_duplicates().merge(first_seen, on='user_id')
activity['day_number'] = (activity.event_date - activity.first_date).dt.days
cohort_size = activity[activity.day_number == 0].user_id.nunique()
retention = []
for day in [1, 7, 14, 30]:
    retained = activity[activity.day_number == day].user_id.nunique()
    retention.append({'day': day, 'retained_users': retained, 'retention': retained / cohort_size})
pd.DataFrame(retention)

## A/B test: view_item -> add_to_cart

In [ ]:
views = df[df.event_name == 'view_item'].groupby('ab_group').user_id.nunique()
carts = df[df.event_name == 'add_to_cart'].groupby('ab_group').user_id.nunique()
ab = pd.DataFrame({'view_users': views, 'cart_users': carts})
ab['conversion'] = ab.cart_users / ab.view_users
ab

In [ ]:
x_a, n_a = ab.loc['A', 'cart_users'], ab.loc['A', 'view_users']
x_b, n_b = ab.loc['B', 'cart_users'], ab.loc['B', 'view_users']
p_a, p_b = x_a / n_a, x_b / n_b
p_pool = (x_a + x_b) / (n_a + n_b)
se = math.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
z = (p_b - p_a) / se
p_value = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))
{'conversion_A': p_a, 'conversion_B': p_b, 'relative_uplift': p_b / p_a - 1, 'z_statistic': z, 'p_value': p_value}